# Music Annotation Project V8.1
## Fold-map resolution and leakage audit

**Runtime:** CPU. Run this notebook before the full-pixel, whole-page OOF notebook.

This notebook resolves the recovered V8 fold artefact correctly:

- the JSON root is expected to contain `sha256` and `rows`;
- `rows` is a **sample-level** table, so repeated pages are expected;
- every sample row must contain `sample_id`, `page`, `source_group`, and `fold`;
- all sample rows for a page must agree on one fold and one source group;
- every source group must belong to exactly one fold;
- multiple recovered files are permitted only when they are semantically identical;
- the derived page-level map is written as a new audit artefact without modifying the source files.

This notebook does **not** train, score, recalibrate, change thresholds, or use clean truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import hashlib
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Search only beneath these experiment roots. Add a root here if a recovered
# fold map is stored elsewhere. Source files remain read-only.
SEARCH_ROOTS = [
    ROOT / 'observed_print_classifier_v8_0b',
    ROOT / 'observed_print_classifier_v8_0c',
    ROOT / 'observed_print_classifier_v8_0d',
    ROOT / 'observed_print_classifier_v8_0e',
    ROOT / 'observed_print_classifier_v8_0f',
]

# Optional exact override. Leave as None to discover every fold_map.json.
FOLD_MAP_OVERRIDE = None

RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S') + '_fold_map_resolution'
OUTPUT_ROOT = ROOT / 'v8_1_full_pixel_oof' / 'fold_map_audit' / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

print('Output:', OUTPUT_ROOT)

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def object_hash(value):
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(',', ':'),
        ensure_ascii=False,
        default=str,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(
        json.dumps(value, indent=2, sort_keys=True, default=str),
        encoding='utf-8',
    )
    tmp.replace(path)


def fail(stage, message, details=None):
    record = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'stage': stage,
        'message': message,
        'details': details,
    }
    atomic_json(OUTPUT_ROOT / 'failure.json', record)
    raise RuntimeError(f'{stage}: {message}')

## Discover recovered fold maps

Discovery is intentionally narrow: only files named `fold_map.json` beneath the configured V8 experiment roots are considered.

In [ ]:
if FOLD_MAP_OVERRIDE is not None:
    candidates = [Path(FOLD_MAP_OVERRIDE)]
else:
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(sorted(root.rglob('fold_map.json')))

# Resolve duplicate path spellings without suppressing distinct files.
unique = {}
for path in candidates:
    if path.exists() and path.is_file():
        unique[str(path.resolve())] = path
candidates = [unique[k] for k in sorted(unique)]

if not candidates:
    fail(
        'discovery',
        'No fold_map.json files were found beneath SEARCH_ROOTS. Set FOLD_MAP_OVERRIDE or add the correct root.',
        [str(x) for x in SEARCH_ROOTS],
    )

print('Recovered fold maps:', len(candidates))
for path in candidates:
    print('-', path)

## Parse and validate each source file

Repeated `page` values are not treated as an error because the authoritative rows are sample-level. The notebook first proves agreement, then collapses the rows to the page level.

In [ ]:
REQUIRED_COLUMNS = ['sample_id', 'page', 'source_group', 'fold']


def load_fold_map(path):
    try:
        obj = json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception as error:
        fail('parse', f'Unreadable JSON: {path}', str(error))

    if not isinstance(obj, dict):
        fail('schema', f'Root must be a JSON object: {path}', type(obj).__name__)

    if 'rows' not in obj or not isinstance(obj['rows'], list):
        fail('schema', f'Expected a list at root["rows"]: {path}', list(obj.keys()))

    frame = pd.DataFrame(obj['rows'])
    missing = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if missing:
        fail('schema', f'Missing required fields in {path}', missing)

    frame = frame[REQUIRED_COLUMNS].copy()

    if frame.empty:
        fail('schema', f'Fold-map rows are empty: {path}')

    if frame[REQUIRED_COLUMNS].isna().any().any():
        nulls = frame[REQUIRED_COLUMNS].isna().sum()
        fail('schema', f'Null required values in {path}', nulls[nulls > 0].to_dict())

    frame['sample_id'] = pd.to_numeric(frame['sample_id'], errors='raise').astype('int64')
    frame['page'] = frame['page'].astype(str).str.strip()
    frame['source_group'] = frame['source_group'].astype(str).str.strip()
    frame['fold'] = pd.to_numeric(frame['fold'], errors='raise').astype('int64')

    if (frame['page'] == '').any() or (frame['source_group'] == '').any():
        fail('schema', f'Blank page or source_group values in {path}')

    duplicate_sample_ids = frame['sample_id'].duplicated(keep=False)
    if duplicate_sample_ids.any():
        q = frame.loc[duplicate_sample_ids].sort_values('sample_id').head(100)
        fail('sample_identity', f'Duplicate sample_id values in {path}', q.to_dict('records'))

    page_fold_counts = frame.groupby('page')['fold'].nunique()
    bad_pages = page_fold_counts[page_fold_counts != 1]
    if len(bad_pages):
        fail('page_fold_consistency', f'Pages assigned to multiple folds in {path}', bad_pages.to_dict())

    page_source_counts = frame.groupby('page')['source_group'].nunique()
    bad_page_sources = page_source_counts[page_source_counts != 1]
    if len(bad_page_sources):
        fail('page_source_consistency', f'Pages assigned to multiple source groups in {path}', bad_page_sources.to_dict())

    source_fold_counts = frame.groupby('source_group')['fold'].nunique()
    bad_sources = source_fold_counts[source_fold_counts != 1]
    if len(bad_sources):
        fail('source_fold_consistency', f'Source groups assigned to multiple folds in {path}', bad_sources.to_dict())

    page_map = (
        frame[['page', 'source_group', 'fold']]
        .drop_duplicates()
        .sort_values(['source_group', 'page', 'fold'])
        .reset_index(drop=True)
    )

    if page_map['page'].duplicated().any():
        fail('page_resolution', f'Page-level map remains non-unique after verified collapse: {path}')

    canonical_rows = (
        frame.sort_values(['sample_id', 'page', 'source_group', 'fold'])
        [REQUIRED_COLUMNS]
        .to_dict('records')
    )
    canonical_pages = page_map.to_dict('records')

    declared_sha = obj.get('sha256')
    return {
        'path': Path(path),
        'file_sha256': sha256_file(path),
        'declared_sha256': declared_sha,
        'sample_rows': int(len(frame)),
        'pages': int(page_map['page'].nunique()),
        'source_groups': int(page_map['source_group'].nunique()),
        'folds': int(page_map['fold'].nunique()),
        'sample_semantic_sha256': object_hash(canonical_rows),
        'page_semantic_sha256': object_hash(canonical_pages),
        'frame': frame,
        'page_map': page_map,
    }

loaded = [load_fold_map(path) for path in candidates]

summary = pd.DataFrame([
    {k: v for k, v in item.items() if k not in {'frame', 'page_map'}}
    for item in loaded
])
display(summary)

## Prove recovered copies are semantically identical

Byte hashes may differ if formatting differs. The authoritative comparison therefore checks the canonical sample rows and the derived page-level assignments. A disagreement fails closed.

In [ ]:
sample_hashes = set(summary['sample_semantic_sha256'])
page_hashes = set(summary['page_semantic_sha256'])

if len(sample_hashes) != 1:
    summary.to_csv(OUTPUT_ROOT / 'recovered_fold_map_comparison.csv', index=False)
    fail(
        'copy_equivalence',
        'Recovered fold maps are not identical at the sample-row semantic level.',
        summary.to_dict('records'),
    )

if len(page_hashes) != 1:
    summary.to_csv(OUTPUT_ROOT / 'recovered_fold_map_comparison.csv', index=False)
    fail(
        'copy_equivalence',
        'Recovered fold maps do not resolve to the same page-level assignments.',
        summary.to_dict('records'),
    )

print('PASS: all recovered fold maps are semantically identical.')

## Write the authoritative derived page map and audit artefacts

The first path in sorted order is selected only as a representative physical copy after semantic equivalence has been proved. The source JSON is not changed.

In [ ]:
selected = loaded[0]
page_map = selected['page_map'].copy()

resolved_csv = OUTPUT_ROOT / 'resolved_page_fold_map.csv'
comparison_csv = OUTPUT_ROOT / 'recovered_fold_map_comparison.csv'
source_group_csv = OUTPUT_ROOT / 'resolved_source_group_fold_map.csv'
resolution_json = OUTPUT_ROOT / 'fold_resolution.json'

page_map.to_csv(resolved_csv, index=False)
summary.to_csv(comparison_csv, index=False)

source_group_map = (
    page_map[['source_group', 'fold']]
    .drop_duplicates()
    .sort_values(['fold', 'source_group'])
    .reset_index(drop=True)
)
source_group_map.to_csv(source_group_csv, index=False)

resolution = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'PASS',
    'resolution_type': 'verified_sample_level_to_page_level_collapse',
    'selected_representative_source': str(selected['path']),
    'selected_representative_file_sha256': selected['file_sha256'],
    'recovered_source_files': [str(x['path']) for x in loaded],
    'recovered_file_sha256': {str(x['path']): x['file_sha256'] for x in loaded},
    'declared_sha256_values': {str(x['path']): x['declared_sha256'] for x in loaded},
    'sample_rows': selected['sample_rows'],
    'resolved_pages': selected['pages'],
    'resolved_source_groups': selected['source_groups'],
    'resolved_folds': selected['folds'],
    'sample_semantic_sha256': selected['sample_semantic_sha256'],
    'page_semantic_sha256': selected['page_semantic_sha256'],
    'resolved_page_csv_sha256': sha256_file(resolved_csv),
    'resolved_source_group_csv_sha256': sha256_file(source_group_csv),
    'assertions': {
        'sample_id_unique': True,
        'one_source_group_per_page': True,
        'one_fold_per_page': True,
        'one_fold_per_source_group': True,
        'recovered_copies_sample_semantically_identical': True,
        'recovered_copies_page_semantically_identical': True,
    },
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'pandas': pd.__version__,
    },
}
atomic_json(resolution_json, resolution)

print(json.dumps(resolution, indent=2))
print('Resolved page map:', resolved_csv)
print('Resolved source-group map:', source_group_csv)
print('Comparison table:', comparison_csv)
print('Resolution record:', resolution_json)

## Optional roster cross-check

If a development score manifest is available, set `DEVELOPMENT_MANIFEST` below and run the cell. The page roster and source-group values must agree exactly. This cell does not alter the resolved fold map.

In [ ]:
DEVELOPMENT_MANIFEST = None
# Example:
# DEVELOPMENT_MANIFEST = ROOT / 'observed_print_classifier_v8_0b' / 'development_score_manifest.csv'

if DEVELOPMENT_MANIFEST is None:
    print('Roster cross-check skipped: DEVELOPMENT_MANIFEST is None.')
else:
    manifest_path = Path(DEVELOPMENT_MANIFEST)
    if not manifest_path.exists():
        fail('roster_cross_check', f'Missing development manifest: {manifest_path}')

    manifest = pd.read_csv(manifest_path)
    required = {'page', 'source_group'}
    if not required.issubset(manifest.columns):
        fail('roster_cross_check', 'Development manifest needs page and source_group.', list(manifest.columns))

    manifest = manifest[['page', 'source_group']].copy()
    manifest['page'] = manifest['page'].astype(str).str.strip()
    manifest['source_group'] = manifest['source_group'].astype(str).str.strip()

    if manifest['page'].duplicated().any():
        fail('roster_cross_check', 'Development manifest page values are not unique.')

    audit = page_map.merge(
        manifest,
        on='page',
        how='outer',
        suffixes=('_fold_map', '_manifest'),
        indicator=True,
    )
    audit['source_group_match'] = (
        audit['source_group_fold_map'] == audit['source_group_manifest']
    )
    audit.to_csv(OUTPUT_ROOT / 'development_manifest_fold_audit.csv', index=False)

    if not (audit['_merge'] == 'both').all():
        fail('roster_cross_check', 'Fold-map and manifest page rosters differ.', audit[audit['_merge'] != 'both'].to_dict('records'))
    if not audit['source_group_match'].all():
        fail('roster_cross_check', 'Fold-map and manifest source_group values differ.', audit[~audit['source_group_match']].to_dict('records'))

    print('PASS: manifest roster and source groups agree exactly with the resolved fold map.')

## Handoff contract for the full-pixel notebook

Use `resolved_page_fold_map.csv` as the page-to-held-out-fold lookup. Keep `fold_resolution.json` and the original `fold_map.json` files as the provenance chain.

The downstream notebook should still assert, before scoring each page:

1. the page exists exactly once in the resolved map;
2. the page source group agrees with the development manifest;
3. the selected model is the model for the page's held-out fold;
4. the model training roster excludes that page and source group;
5. all non-PRINT output pixels reproduce frozen V7.8 exactly.

In [ ]:
print('FOLD-MAP RESOLUTION AUDIT COMPLETE')
print('Authoritative downstream lookup:', resolved_csv)
print('Audit status: PASS')